# EX_11 — RAG avanzado / agéntico (ejercicios)

**Notebook de referencia:** `notebook/11_RAG_Avanzado_Agentico.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Self-RAG checklist

Lista 3 preguntas binarias que el sistema podría autoevaluar sobre un borrador (soporte en contexto, completitud, seguridad). Respuesta en markdown.


**Checklist Self-RAG — 3 preguntas binarias sobre un borrador de respuesta:**

1. **¿Soporte en contexto?** — ¿Cada afirmación factual del borrador aparece explícita o deducible **solo** de los fragmentos recuperados `[1]…[n]`? → **SÍ / NO**

2. **¿Completitud?** — ¿El borrador responde **todos** los aspectos de la pregunta del usuario sin omitir requisitos clave (tipos de IA, aplicaciones, limitaciones)? → **SÍ / NO**

3. **¿Seguridad / abstención?** — Si el contexto es insuficiente o ambiguo, ¿el borrador **se abstiene** de inventar datos y declara la limitación en lugar de alucinar? → **SÍ / NO**

> Si alguna respuesta es **NO**, el grafo agentic debería re-enrutar a `retrieve` (reformular query), `generate` (reescribir) o `finalize` con mensaje de no información.


## Actividad 2 — Grafo textual

Dibuja en ASCII o Mermaid (en markdown) un flujo con nodos: `plan`, `retrieve`, `grade_docs`, `generate`, `finalize`.


```mermaid
flowchart TD
    START([Inicio]) --> plan[plan]
    plan --> retrieve[retrieve]
    retrieve --> grade_docs[grade_docs]
    grade_docs -->|docs relevantes| generate[generate]
    grade_docs -->|docs irrelevantes| plan
    generate --> grade_answer{¿Borrador OK?}
    grade_answer -->|SÍ| finalize[finalize]
    grade_answer -->|NO| plan
    finalize --> END([Respuesta final])

    style plan fill:#e8f4fd
    style grade_docs fill:#fff3cd
    style finalize fill:#d4edda
```

**Lectura del flujo:**

| Nodo | Función |
|------|---------|
| `plan` | Clasifica la pregunta y decide estrategia (reformular query, número de pasos). |
| `retrieve` | Busca chunks en FAISS con la query (original o reformulada). |
| `grade_docs` | Evalúa relevancia de cada documento (Self-RAG: ¿útil para responder?). |
| `generate` | Produce borrador fundamentado en docs aprobados. |
| `finalize` | Aplica checklist Self-RAG, añade citas `[n]` y devuelve respuesta final. |


## Actividad 3 — Citas

Escribe una plantilla de prompt que **obligue** a citar fragmentos del contexto con identificadores `[n]` al responder.


In [1]:
def build_citation_prompt(context_chunks: list[str], question: str) -> str:
    """Construye prompt con fragmentos numerados [1]..[n] y reglas de citación obligatoria."""
    numbered = "\n\n".join(
        f"[{i}] {chunk.strip()}" for i, chunk in enumerate(context_chunks, start=1)
    )
    return f"""Eres un asistente RAG sobre inteligencia artificial. Responde ÚNICAMENTE usando los fragmentos numerados del contexto.

Reglas de citación (OBLIGATORIAS):
- Cada afirmación factual debe ir seguida de la cita del fragmento, p. ej. "La IA débil está diseñada para tareas específicas [1]."
- Usa solo identificadores [1], [2], ... presentes abajo. No inventes números.
- Si ningún fragmento responde la pregunta, escribe: "No tengo información suficiente en el contexto."
- No añadas conocimiento externo.

=== CONTEXTO ===
{numbered}

=== PREGUNTA ===
{question}

=== RESPUESTA (con citas [n] en cada hecho) ===
"""


citation_prompt = build_citation_prompt(
    context_chunks=[
        "La IA débil (Narrow AI) está diseñada para tareas específicas como reconocimiento de voz o sistemas de recomendación.",
        "El procesamiento de lenguaje natural permite a los sistemas entender, interpretar y generar texto humano.",
        "Aplicaciones comunes de la IA incluyen visión por computadora, NLP y sistemas de recomendación.",
    ],
    question="¿Qué es la IA débil y qué aplicaciones de IA se mencionan en el contexto?",
)

print(citation_prompt)


Eres un asistente RAG. Responde ÚNICAMENTE usando los fragmentos numerados del contexto.

Reglas de citación (OBLIGATORIAS):
- Cada afirmación factual debe ir seguida de la cita del fragmento, p. ej. "El IBI se paga anualmente [1]."
- Usa solo identificadores [1], [2], ... presentes abajo. No inventes números.
- Si ningún fragmento responde la pregunta, escribe: "No tengo información suficiente en el contexto."
- No añadas conocimiento externo.

=== CONTEXTO ===
[1] El IBI se paga anualmente y grava bienes inmuebles urbanos y rústicos.

[2] Bonificaciones IBI: familias numerosas hasta 90%, energías renovables 50%.

=== PREGUNTA ===
¿Cuándo se paga el IBI y qué bonificaciones existen?

=== RESPUESTA (con citas [n] en cada hecho) ===

